# Task 4 — Final Hold-out Benchmark

## Objective

Compare the five saved representation-learning models on the untouched outer test split, using **Precision@1/5/10**, **Recall@1/5/10**, and **mAP@10**.

**Evaluation protocol.** `splits/task4/test.csv` supplies labelled queries, while each model's saved gallery embeddings in `artifacts/task4/models/` supply the retrieval candidates. A retrieved item is relevant when its `articleType__gender` label matches the query label. Test labels are recreated from the persisted, train-fitted label encoder. `train.csv` is read only to attach labels to the saved gallery IDs; it is never used as a query set or to fit any preprocessing or model component.

Test images are located in the original training-image directory because this is an outer hold-out split, not the separate unlabelled prediction set. The notebook deliberately uses raw cosine retrieval for every model, isolating the learned embeddings from any optional reranker.

## 1. Setup and reproducibility

The common preprocessing contract is loaded from `artifacts/task4/configs/image_preprocessing.json`. Metric-learning models receive letterbox resize, tensor conversion, and training-set RGB normalisation; the autoencoder receives the tensor-only reconstruction transform used when it was trained.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while (
    PROJECT_ROOT != PROJECT_ROOT.parent
    and not (PROJECT_ROOT / "pyproject.toml").exists()
):
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import json
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from PIL import Image

from src.task4.config import (
    ARTIFACT_DIR,
    CONFIG_DIR,
    IMAGE_DIR,
    MODEL_DIR,
    SEED,
    SPLIT_DIR,
)
from src.task4.data import make_evaluation_loader
from src.task4.models import ConvolutionalAutoencoder, ResNet18Encoder
from src.task4.preprocessing import cae_transform, metric_eval_transform
from src.task4.retrieval import extract_embeddings, search_cosine
from src.task4.training import DEVICE, set_seed

set_seed(SEED)
pd.set_option("display.float_format", lambda value: f"{value:.4f}")

MODEL_NAMES = ("cae", "triplet", "supcon", "multi_similarity", "arcface")
PRIMARY_METRIC = "mAP@10"
BENCHMARK_DIR = ARTIFACT_DIR / "benchmark"
print(f"Device: {DEVICE}")

Device: cpu


## 2. Verify inputs and construct the labelled test set

The preflight requires a checkpoint, saved gallery embeddings, and their matching gallery IDs for every candidate model. Test labels are recreated from test metadata using the persisted train-fitted encoder. Encoder-unseen test labels are removed and counted explicitly. The outer training split is then read solely to map saved gallery IDs to their encoder label IDs, which is required to determine retrieval relevance.

In [ ]:
required_files = ("best.pt", "gallery_embeddings.npy", "gallery_ids.npy")
artifact_rows = []
for model_name in MODEL_NAMES:
    model_dir = MODEL_DIR / model_name
    missing = [name for name in required_files if not (model_dir / name).exists()]
    artifact_rows.append(
        {
            "Model": model_name,
            "Ready": not missing,
            "Missing artifacts": ", ".join(missing) if missing else "",
        }
    )

artifact_status = pd.DataFrame(artifact_rows)
display(artifact_status)

missing_models = artifact_status.loc[~artifact_status["Ready"], "Model"].tolist()
if missing_models:
    raise FileNotFoundError(
        "Final benchmark requires all five saved model bundles. Missing: "
        + ", ".join(missing_models)
    )

if not IMAGE_DIR.exists():
    raise FileNotFoundError(f"Hold-out images are not available at {IMAGE_DIR}")

In [ ]:
with (CONFIG_DIR / "articleType_gender_label_encoder.json").open(
    encoding="utf-8"
) as file:
    label_encoder = json.load(file)
label_to_index = label_encoder["label_to_index"]
classes = np.asarray(label_encoder["classes"], dtype=object)
LABEL_COLUMN = label_encoder["label_column"]
LABEL_ID_COLUMN = label_encoder["label_id_column"]
separator = label_encoder["separator"]

outer_train_df = pd.read_csv(SPLIT_DIR / "train.csv")
outer_train_df[LABEL_COLUMN] = (
    outer_train_df["articleType"].str.strip()
    + separator
    + outer_train_df["gender"].str.strip()
)
outer_train_df[LABEL_ID_COLUMN] = outer_train_df[LABEL_COLUMN].map(label_to_index)

test_df = pd.read_csv(SPLIT_DIR / "test.csv")
test_df[LABEL_COLUMN] = (
    test_df["articleType"].str.strip() + separator + test_df["gender"].str.strip()
)
test_df[LABEL_ID_COLUMN] = test_df[LABEL_COLUMN].map(label_to_index)


In [ ]:
test_rows_before_filtering = len(test_df)
unseen_class_mask = test_df[LABEL_ID_COLUMN].isna()
removed_unseen_rows = int(unseen_class_mask.sum())
test_df = test_df.loc[~unseen_class_mask].copy()
test_df[LABEL_ID_COLUMN] = test_df[LABEL_ID_COLUMN].astype("int64")

print(f"Hold-out test rows before filtering: {test_rows_before_filtering:,}")
print(f"Rows removed for encoder-unseen classes: {removed_unseen_rows:,}")
print(f"Seen-class test rows retained: {len(test_df):,}")


Hold-out test rows before filtering: 3,777
Rows removed for encoder-unseen classes: 3
Seen-class test rows retained: 3,774


## 3. Shared scoring helpers

The helper returns each query's contribution as well as aggregate metrics. Gallery support is the number of same-label candidates available in that model's saved gallery.

In [ ]:
def build_model(model_name: str):
    if model_name == "cae":
        return ConvolutionalAutoencoder(ResNet18Encoder())
    return ResNet18Encoder()


def load_checkpoint(model_name: str):
    checkpoint_path = MODEL_DIR / model_name / "best.pt"
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)

    model = build_model(model_name)
    state_dict = checkpoint["model_state_dict"]
    # ArcFace checkpoints may include the training-only classifier loss.
    if model_name == "arcface":
        state_dict = {
            key: value
            for key, value in state_dict.items()
            if not key.startswith("arcface_loss.")
        }
    model.load_state_dict(state_dict, strict=True)
    return model.to(DEVICE).eval(), checkpoint


def per_query_metrics(ranked_indices, query_labels, gallery_labels, k=10):
    ranked_labels = gallery_labels[ranked_indices[:, :k]]
    relevant = ranked_labels == query_labels[:, None]
    ranks = np.arange(1, relevant.shape[1] + 1)
    gallery_counts = Counter(gallery_labels.tolist())
    support = np.asarray([gallery_counts[int(label)] for label in query_labels])
    if np.any(support < 1):
        raise ValueError(
            "Every scored query must have at least one same-label gallery item"
        )

    return pd.DataFrame(
        {
            "Precision@1": relevant[:, :1].mean(axis=1),
            "Precision@5": relevant[:, :5].mean(axis=1),
            "Precision@10": relevant.mean(axis=1),
            "Recall@1": relevant[:, :1].sum(axis=1) / support,
            "Recall@5": relevant[:, :5].sum(axis=1) / support,
            "Recall@10": relevant.sum(axis=1) / support,
            "mAP@10": (np.cumsum(relevant, axis=1) / ranks * relevant).sum(axis=1)
            / np.minimum(support, 10),
            "gallery_support": support,
        }
    )


def bootstrap_mean_ci(values, n_resamples=1_000, seed=SEED):
    values = np.asarray(values, dtype=float)
    rng = np.random.default_rng(seed)
    bootstrap_means = rng.choice(
        values, size=(n_resamples, len(values)), replace=True
    ).mean(axis=1)
    return np.quantile(bootstrap_means, [0.025, 0.975])


def bootstrap_difference_ci(left, right, n_resamples=1_000, seed=SEED):
    differences = np.asarray(left, dtype=float) - np.asarray(right, dtype=float)
    return bootstrap_mean_ci(differences, n_resamples=n_resamples, seed=seed)

## 4. Extract test embeddings and evaluate all five models

The gallery embeddings and IDs are loaded from each model's saved artifact directory. `train.csv` is used only to map those gallery IDs to class IDs. Each model applies its original evaluation preprocessing to the seen-class hold-out test images before their query embeddings are extracted and searched against its saved gallery.

In [ ]:
gallery_label_by_id = outer_train_df.set_index("id")[LABEL_ID_COLUMN]

model_outputs = {}
summary_rows = []
coverage_rows = []

for model_name in MODEL_NAMES:
    model_dir = MODEL_DIR / model_name
    gallery_embeddings = np.load(model_dir / "gallery_embeddings.npy").astype("float32")
    gallery_ids = np.load(model_dir / "gallery_ids.npy").astype("int64")
    if len(gallery_embeddings) != len(gallery_ids):
        raise ValueError(
            f"{model_name}: gallery embeddings and IDs must have equal length"
        )

    gallery_labels = gallery_label_by_id.reindex(gallery_ids)
    if gallery_labels.isna().any():
        raise ValueError(f"{model_name}: a saved gallery ID is absent from train.csv")
    gallery_labels = gallery_labels.to_numpy(dtype="int64")

    evaluation_df = test_df[test_df[LABEL_ID_COLUMN].isin(gallery_labels)].copy()
    excluded_ungalleryable_queries = len(test_df) - len(evaluation_df)
    if evaluation_df.empty:
        raise ValueError(f"{model_name}: no test queries have a matching gallery class")

    model, checkpoint = load_checkpoint(model_name)
    expected_size = checkpoint["model_config"].get("input_size")
    if expected_size != [128, 128]:
        raise ValueError(
            f"{model_name}: unexpected checkpoint input size {expected_size}"
        )

    query_loader = make_evaluation_loader(
        model_name,
        evaluation_df,
        cae_transform,
        metric_eval_transform,
        IMAGE_DIR,
    )
    query_embeddings, query_ids, query_labels = extract_embeddings(model, query_loader)
    _, ranked_indices, _ = search_cosine(
        query_embeddings, gallery_embeddings, maximum_k=10
    )
    query_scores = per_query_metrics(ranked_indices, query_labels, gallery_labels)

    aggregate = query_scores.drop(columns="gallery_support").mean().to_dict()
    aggregate["Model"] = model_name
    summary_rows.append(aggregate)
    coverage_rows.append(
        {
            "Model": model_name,
            "Scored queries": len(query_ids),
            "Excluded encoder-unseen queries": removed_unseen_rows,
            "Excluded absent-gallery queries": excluded_ungalleryable_queries,
            "Gallery size": len(gallery_ids),
        }
    )
    model_outputs[model_name] = {
        "gallery_ids": gallery_ids,
        "gallery_labels": gallery_labels,
        "query_ids": query_ids,
        "query_labels": query_labels,
        "ranked_indices": ranked_indices,
        "query_scores": query_scores,
    }

coverage_table = pd.DataFrame(coverage_rows).set_index("Model")
display(coverage_table)

## 5. Required comparison table

All seven requested metrics are reported as proportions; larger is better. `mAP@10` is the primary selection metric because it rewards placing multiple relevant items near the top, not merely obtaining a single correct result.

In [ ]:
metric_columns = [
    "Precision@1",
    "Precision@5",
    "Precision@10",
    "Recall@1",
    "Recall@5",
    "Recall@10",
    "mAP@10",
]
comparison_table = (
    pd.DataFrame(summary_rows)
    .set_index("Model")
    .loc[:, metric_columns]
    .sort_values(PRIMARY_METRIC, ascending=False)
)
BENCHMARK_DIR.mkdir(parents=True, exist_ok=True)
comparison_table.to_csv(BENCHMARK_DIR / "holdout_comparison.csv")
display(comparison_table.style.format("{:.4f}").highlight_max(axis=0, color="#d9ead3"))

winner = comparison_table.index[0]
print(f"Leading model by {PRIMARY_METRIC}: {winner}")

## 6. Decision-support analysis: uncertainty and long-tail behaviour

These analyses are useful evidence for the report beyond a single point estimate:

- **Bootstrap 95% confidence intervals** show whether apparent mAP@10 differences are stable across hold-out queries.
- **Gallery-support bands** show whether a model's quality holds for infrequent product categories, rather than being driven only by common categories.

In [ ]:
interval_rows = []
for model_name in comparison_table.index:
    scores = model_outputs[model_name]["query_scores"]["mAP@10"]
    lower, upper = bootstrap_mean_ci(scores)
    interval_rows.append(
        {
            "Model": model_name,
            "mAP@10": scores.mean(),
            "95% CI lower": lower,
            "95% CI upper": upper,
        }
    )
interval_table = pd.DataFrame(interval_rows).set_index("Model")
interval_table.to_csv(BENCHMARK_DIR / "map_at_10_confidence_intervals.csv")
display(interval_table.style.format("{:.4f}"))

runner_up = comparison_table.index[1]
winner_ci = bootstrap_difference_ci(
    model_outputs[winner]["query_scores"]["mAP@10"],
    model_outputs[runner_up]["query_scores"]["mAP@10"],
)
print(
    f"Paired {winner} − {runner_up} mAP@10 difference: "
    f"{comparison_table.loc[winner, PRIMARY_METRIC] - comparison_table.loc[runner_up, PRIMARY_METRIC]:.4f} "
    f"(95% bootstrap CI {winner_ci[0]:.4f} to {winner_ci[1]:.4f})"
)

In [ ]:
support_bins = [0, 4, 19, 99, np.inf]
support_labels = ["1–4", "5–19", "20–99", "100+"]
tail_rows = []
for model_name, output in model_outputs.items():
    scores = output["query_scores"].copy()
    scores["Gallery support"] = pd.cut(
        scores["gallery_support"], bins=support_bins, labels=support_labels
    )
    for support_band, group in scores.groupby("Gallery support", observed=False):
        if len(group):
            tail_rows.append(
                {
                    "Model": model_name,
                    "Gallery support": support_band,
                    "Queries": len(group),
                    "mAP@10": group["mAP@10"].mean(),
                }
            )

long_tail_table = (
    pd.DataFrame(tail_rows)
    .pivot(index="Model", columns="Gallery support", values="mAP@10")
    .reindex(comparison_table.index)
)
long_tail_table.to_csv(BENCHMARK_DIR / "map_at_10_by_gallery_support.csv")
display(long_tail_table.style.format("{:.4f}"))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
plot_values = comparison_table[PRIMARY_METRIC].sort_values()
ax.barh(plot_values.index, plot_values.values, color="#4c78a8")
ax.set_xlabel("mAP@10")
ax.set_title("Hold-out retrieval benchmark (higher is better)")
ax.set_xlim(0, min(1.0, max(0.05, plot_values.max() * 1.15)))
ax.grid(axis="x", alpha=0.25)
for y, value in enumerate(plot_values.values):
    ax.text(value, y, f" {value:.3f}", va="center")
fig.tight_layout()
fig.savefig(
    BENCHMARK_DIR / "holdout_map_at_10_comparison.png", dpi=160, bbox_inches="tight"
)
plt.show()

## 7. Qualitative retrieval audit

Metrics should be paired with a small visual audit. The cell below displays a random set of hold-out queries and the winning model's top-five gallery results. In the report, use this to discuss a few representative successes and failures (for example: visually similar but wrong category, gender ambiguity, or rare categories), rather than claiming that the images are quantitative evidence.

In [ ]:
def show_retrieval_examples(model_name=winner, n_queries=3, k=5, seed=SEED):
    output = model_outputs[model_name]
    rng = np.random.default_rng(seed)
    chosen = rng.choice(
        len(output["query_ids"]),
        size=min(n_queries, len(output["query_ids"])),
        replace=False,
    )
    fig, axes = plt.subplots(
        len(chosen), k + 1, figsize=(2.2 * (k + 1), 2.5 * len(chosen))
    )
    axes = np.atleast_2d(axes)

    for row, query_index in enumerate(chosen):
        query_id = output["query_ids"][query_index]
        query_label = classes[output["query_labels"][query_index]]
        gallery_indices = output["ranked_indices"][query_index, :k]
        ids = [query_id, *output["gallery_ids"][gallery_indices]]
        labels = [query_label, *classes[output["gallery_labels"][gallery_indices]]]
        for column, (item_id, item_label) in enumerate(zip(ids, labels)):
            with Image.open(IMAGE_DIR / f"{int(item_id)}.jpg") as image:
                axes[row, column].imshow(image.convert("RGB"))
            axes[row, column].axis("off")
            if column == 0:
                axes[row, column].set_title(f"Query\n{item_label}", fontsize=8)
            else:
                is_relevant = item_label == query_label
                axes[row, column].set_title(
                    f"#{column} {'✓' if is_relevant else '✗'}\n{item_label}", fontsize=8
                )
    fig.suptitle(f"{model_name}: hold-out queries and top-{k} gallery results", y=1.02)
    fig.tight_layout()
    plt.show()


show_retrieval_examples()

## What to present

Keep the main report compact: the required comparison table, the top-model mAP@10 confidence interval (and paired difference to the runner-up), and one long-tail table/figure are strong complementary evidence. Add two or three qualitative retrieval examples only to explain observed failure modes. State the query coverage from Section 4 so the reader knows that cold-start labels were not hidden inside the closed-set scores.

For the final recommendation, weigh accuracy, uncertainty, long-tail behaviour, qualitative errors, inference cost, and implementation complexity—not just the single largest mAP@10 value.